# VaLPAS based target identification

## Front matter & Imports

In [1]:
# iPython magic to autoreload modules everytime code is executed to propagate changes to the code
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os

import pandas as pd
from valpas.valpas_core import associate


os.makedirs('./tmp', exist_ok=True)
os.makedirs('./output', exist_ok=True)

## Harmonization of Yarrowia sample ids

The sample ids across protomics and metabolomics / lipidomics are currently not matching up. To prepare the data tables for VaLPAS we first need to harmonize the sample ids.

The code below relies on 3 files contained in the `data` directory:
- ylip-sample_mapping.tsv
  - Contains the mapping of sample ids between proteomics and metabolomics
  - extracted from `PPI_sampleInfo_ylp.xlsx`, sheet `sampleNames`, columns `Global_REDOX_ProteomicNames` & `Metabolomic`
- ylip-proteomics.tsv
  - data table containing the proteomics abundances
  - extracted from `PPI_yarrowia_protein_abundance.csv`
  - columns extracted from the excel file are the abundance columns and the `Protein` id column. Any additional metadata columns are not included.
- ylip-metabolomics.tsv
  - data table contating the metabolomics abundances
  - extracted from `YLIP_Metabolomics_ALL_Modes_Pmart_normalized.xlsx`
  - 1to1 copy of data from `Sheet1`

### Import of datasets

#### Sample ID mapping table

In [2]:
sample_mapping = pd.read_csv(
    filepath_or_buffer='./data/ylip-sample_mapping.tsv',
    delimiter='\t',
)

sample_mapping.head()

,proteomics_sample_id,metabolomics_sample_id
0,Biostat2L_29_Oscillation_3.5,PPI_Ylip_Osc_3hr_01
1,Biostat2L_29_Oscillation_4.5,PPI_Ylip_Osc_4hr_02
2,Biostat2L_29_Oscillation_5.5,PPI_Ylip_Osc_5hr_03
3,Biostat2L_29_Oscillation_24,PPI_Ylip_Osc_24hr_04
4,Biostat2L_29_Oscillation_48,PPI_Ylip_Osc_47hr_05


#### Proteomics abundances table

In [3]:
proteomics = pd.read_csv(
    filepath_or_buffer='./data/ylip-proteomics.tsv',
    delimiter='\t',
)
proteomics.head()

,Protein,Biostat2L_29_Oscillation_24,Biostat2L_29_Oscillation_3.5,Biostat2L_29_Oscillation_4.5,Biostat2L_29_Oscillation_48,Biostat2L_29_Oscillation_5.5,Biostat2L_29_Oscillation_72,Biostat2L_30_Control_24,Biostat2L_30_Control_3.5,Biostat2L_30_Control_4.5,...,Biostat2L_33_Low_4.5,Biostat2L_33_Low_48,Biostat2L_33_Low_5.5,Biostat2L_33_Low_72,Biostat2L_34_Low_24,Biostat2L_34_Low_3.5,Biostat2L_34_Low_4.5,Biostat2L_34_Low_48,Biostat2L_34_Low_5.5,Biostat2L_34_Low_72
0,jgi|YarliW29_1|100,25.770213,25.369272,25.511412,25.649494,25.666597,25.495693,25.457543,25.620312,25.624321,...,25.647082,26.254312,25.752609,26.329296,26.010065,25.348436,25.474421,26.227300,25.636968,26.059671
1,jgi|YarliW29_1|1000,17.293959,18.245944,18.186630,16.996140,18.011891,16.940736,17.495514,18.042939,18.566494,...,18.061817,17.550454,18.012434,17.492877,17.668527,18.018299,18.046325,17.466942,18.003958,17.402518
2,jgi|YarliW29_1|1002,22.817965,23.813276,25.302130,22.221128,25.155731,22.204025,22.626837,24.269189,25.312090,...,25.410495,23.148530,25.186652,22.770328,23.563942,24.055592,25.125115,22.947880,25.069054,22.543373
3,jgi|YarliW29_1|1003,21.661173,22.834460,22.811937,21.343351,22.638917,21.351494,21.925310,22.795371,22.970198,...,22.730409,21.646451,22.740655,21.371498,21.903156,22.757346,22.752821,21.644129,22.521363,21.391500
4,jgi|YarliW29_1|1004,24.612309,23.180487,23.514879,26.057785,23.618234,26.347074,25.556813,23.298582,23.559677,...,23.418239,23.387934,23.352792,24.626466,23.622010,23.420203,23.752784,24.104213,23.876573,24.727999


#### Metabolomics abundances table

In [4]:
metabolomics = pd.read_csv(
    filepath_or_buffer='./data/ylip-metabolomics.tsv',
    delimiter='\t',
)
metabolomics.rename(columns={'Metabolite name': 'Metabolite'}, inplace=True)
metabolomics.head()

,Metabolite,PPI_Ylip_Ctrl_5hr_35,PPI_Ylip_Low_48hr_53,PPI_Ylip_Ctrl_5hr_11,PPI_Ylip_Ctrl_24hr_37,PPI_Ylip_Low_48hr_47,PPI_Ylip_Osc_72hr_06,PPI_Ylip_Low_72hr_54,PPI_Ylip_Low_4hr_34,PPI_Ylip_Osc_47hr_05,...,PPI_Ylip_Low_3hr_49,PPI_Ylip_Osc_48hr_28,PPI_Ylip_Ctrl_24hr_25,PPI_Ylip_Osc_24hr_04,PPI_Ylip_Ctrl_3hr_19,PPI_Ylip_Ctrl_5hr_23,PPI_Ylip_Ctrl_72hr_41,PPI_Ylip_Ctrl_3hr_31,PPI_Ylip_Osc_5hr_03,PPI_Ylip_Low_4hr_44
0,[6]-Gingerol_RP,25.304971,25.922555,25.548379,25.298681,25.780203,23.771397,24.708991,25.304508,25.934016,...,25.604529,25.859027,25.125309,25.514227,25.261703,25.421335,26.255295,23.357804,25.796217,25.200045
1,"1-(2-hydroxy-4,6-dimethoxyphenyl)ethanone_RP",20.994103,24.758278,20.993220,20.874734,22.332716,24.636485,25.891304,21.144469,22.058745,...,21.374189,22.293956,23.950306,24.495085,21.737653,20.314038,25.083603,26.935488,19.514575,20.827261
2,1-Monostearin_RP,27.830119,21.642454,21.854335,21.388737,26.363149,29.145346,23.008059,22.373570,26.187290,...,27.130488,25.472312,20.366222,25.268657,27.821596,27.801279,21.219226,20.798368,28.655229,27.910641
3,"1,3-diphenylguanidine_RP",18.949790,19.819692,17.945497,18.782512,20.989119,17.775612,20.088038,18.900946,19.689460,...,18.695562,21.381499,18.484466,18.746473,18.369038,18.596721,23.473059,21.450317,19.730435,19.283907
4,10-HYDROXYDECANOATE_RP,23.743140,24.410736,24.345803,24.103621,24.545223,25.108104,24.367260,24.045975,24.931918,...,24.438505,24.529272,23.915051,24.350315,24.608141,24.176449,25.230456,23.840167,24.011480,23.716725


### Renaming of proteomics sample ids to metabolomics sample ids

To achieve this we create a dictionary from the sample id mapping table where the `key = proteomics_sample_id` & `value = metabolomics_sample_id`. The we rename the column index using the established mapping dictionary. 

In [5]:
column_mapping = dict(zip(sample_mapping['proteomics_sample_id'], sample_mapping['metabolomics_sample_id']))
proteomics.rename(columns=column_mapping, inplace=True)
proteomics.head()

,Protein,PPI_Ylip_Osc_24hr_04,PPI_Ylip_Osc_3hr_01,PPI_Ylip_Osc_4hr_02,PPI_Ylip_Osc_47hr_05,PPI_Ylip_Osc_5hr_03,PPI_Ylip_Osc_72hr_06,PPI_Ylip_Ctrl_24hr_13,PPI_Ylip_Ctrl_3hr_07,PPI_Ylip_Ctrl_4hr_09,...,PPI_Ylip_Low_4hr_44,PPI_Ylip_Low_48hr_47,PPI_Ylip_Low_5hr_45,PPI_Ylip_Low_72hr_48,PPI_Ylip_Low_24hr_52,PPI_Ylip_Low_3hr_49,PPI_Ylip_Low_4hr_50,PPI_Ylip_Low_48hr_53,PPI_Ylip_Low_5hr_51,PPI_Ylip_Low_72hr_54
0,jgi|YarliW29_1|100,25.770213,25.369272,25.511412,25.649494,25.666597,25.495693,25.457543,25.620312,25.624321,...,25.647082,26.254312,25.752609,26.329296,26.010065,25.348436,25.474421,26.227300,25.636968,26.059671
1,jgi|YarliW29_1|1000,17.293959,18.245944,18.186630,16.996140,18.011891,16.940736,17.495514,18.042939,18.566494,...,18.061817,17.550454,18.012434,17.492877,17.668527,18.018299,18.046325,17.466942,18.003958,17.402518
2,jgi|YarliW29_1|1002,22.817965,23.813276,25.302130,22.221128,25.155731,22.204025,22.626837,24.269189,25.312090,...,25.410495,23.148530,25.186652,22.770328,23.563942,24.055592,25.125115,22.947880,25.069054,22.543373
3,jgi|YarliW29_1|1003,21.661173,22.834460,22.811937,21.343351,22.638917,21.351494,21.925310,22.795371,22.970198,...,22.730409,21.646451,22.740655,21.371498,21.903156,22.757346,22.752821,21.644129,22.521363,21.391500
4,jgi|YarliW29_1|1004,24.612309,23.180487,23.514879,26.057785,23.618234,26.347074,25.556813,23.298582,23.559677,...,23.418239,23.387934,23.352792,24.626466,23.622010,23.420203,23.752784,24.104213,23.876573,24.727999


#### Filtering the Metabolomics data 

In [6]:
metabolomics_sorted = metabolomics.set_index('Metabolite').sort_index(axis=1)
for group in ['Ctrl', 'Osc']:
    filtered_df = metabolomics_sorted.filter(axis=1, like=group)
    filtered_df.reset_index().to_csv(f'./tmp/ylip-{group.lower()}__metabolomics.csv', sep=',', index=False)


#### Filtering the Proteomics data

In [7]:
proteomics_sorted = proteomics.set_index('Protein').sort_index(axis=1)
for group in ['Ctrl', 'Osc']:
    filtered_df = proteomics_sorted.filter(axis=1, like=group)
    filtered_df.reset_index().to_csv(f'./tmp/ylip-{group.lower()}__proteomics.csv', sep=',', index=False)


## Running the VaLPAS analysis

The first thing we need to do is generate the association values between metabolites and proteins for each set.
We are going to look at the following association metrics:
- pearson

Furthermore we will generate the assocations mentioned above for the following condition classes:
- Oscilating oxygen (`osc`)
- Control (`ctrl`)

Ergo in total we will generate 4 datasets.

The code cell below will itterate through all combinations outlined above and generate the outfiles.

In [8]:

for type_ in ['osc', 'ctrl']:
    fpath1 = Path(f'./tmp/ylip-{type_}__proteomics.csv')
    fpath2 = Path(f'./tmp/ylip-{type_}__metabolomics.csv')
    fpath_out = Path(f'./output/ylip-{type_}_out_pearson.csv')

    association_result = associate(
        association_type='pearson',
        infile=fpath1,
        infile2=fpath2,
        file_type='csv',
        filter_cutoff=0.9,
        outfile=fpath_out,
        output_type='sorted_list',
        overwrite_output=True
    )

### Calculating the delta association score between control and oscillating oxygen
Now that the association values between metabolites and proteins are calculated we will generate the difference between association scores between the control environment and oscilating oxygen environments.
For easier calculations we will define a function that takes the association list from two conditions as well as a defined association metric and return a data frame or list that contains all of the deltas.

In [9]:
def gen_association_deltas(
        in1: str,
        in2: str,
        left: str = '_x',
        right: str = '_y'
        ):
    """
    _summary_

    Parameters
    ----------
    in1 : str
        _description_
    in2 : str
        _description_
    left : str, optional
        _description_, by default '_x'
    right : str, optional
        _description_, by default '_y'

    Returns
    -------
    _type_
        _description_
    """
    
    # Importing of the association lists
    # Currently valpas produces duplicate entries in the imported lists.
    # The `.drop_duplicates` can be removed once that is fixed.
    df1 = pd.read_csv(
        filepath_or_buffer=in1,
    ).drop_duplicates()
    df2 = pd.read_csv(
        filepath_or_buffer=in2
    ).drop_duplicates()    

    # merging the two lists with optional suffix definition
    df = df1.merge(
        right=df2,
        how='outer',
        on=['proteomics','metabolomics'],
        suffixes=[left, right],
        validate="1:1",
        )
    
    # calculating the delta (left -> right) & absolute delta
    df['delta'] = -(df[f'association{left}'] - df[f'association{right}'])
    df['delta_abs'] = abs(df['delta'])

    return df

def gen_delta_df(association_type, cond_1, cond_2):
    fpath_in1 = f'./output/ylip-{cond_1}_out_{association_type}.csv'
    fpath_in2 = f'./output/ylip-{cond_2}_out_{association_type}.csv'
    df = gen_association_deltas(fpath_in1, fpath_in2, left=f'_{cond_1}', right=f'_{cond_2}')
    df = df.sort_values('delta_abs', ascending=False)
    return df

Now we can generate a delta list for the pearson correlation lists between `control` and `osc`.

In [10]:
df_ctrl_v_osc = gen_delta_df('pearson', 'ctrl', 'osc')
df_ctrl_v_osc.head(20)

,proteomics,metabolomics,association_ctrl,counts_ctrl,association_osc,counts_osc,delta,delta_abs
1035001,"NG,NG-Dimethyl-L-arginine_HN",jgi|YarliW29_1|2159,0.960875,18.0,-0.958126,18.0,-1.919002,1.919002
188094,"4,6-Dihydroxypyrimidine_HN",jgi|YarliW29_1|2159,0.961688,18.0,-0.953633,18.0,-1.915321,1.915321
433546,D-(-)-Quinic acid_HN,jgi|YarliW29_1|2015,-0.955142,18.0,0.959837,18.0,1.914979,1.914979
310451,Ala-Gly_HN,jgi|YarliW29_1|4796,-0.948175,18.0,0.963573,18.0,1.911748,1.911748
309453,Ala-Gly_HN,jgi|YarliW29_1|2038,-0.968872,18.0,0.942664,18.0,1.911535,1.911535
491531,DL-2-Methylglutamic acid_HN,jgi|YarliW29_1|2159,-0.948822,18.0,0.960350,18.0,1.909172,1.909172
1034964,"NG,NG-Dimethyl-L-arginine_HN",jgi|YarliW29_1|2038,-0.961269,18.0,0.944150,18.0,1.905420,1.905420
434614,D-(-)-Quinic acid_HN,jgi|YarliW29_1|4933,0.930527,18.0,-0.974560,18.0,-1.905086,1.905086
434479,D-(-)-Quinic acid_HN,jgi|YarliW29_1|4631,0.965576,18.0,-0.938717,18.0,-1.904293,1.904293
469413,D-Ribose 1-phosphate_HN,jgi|YarliW29_1|2015,0.955881,18.0,-0.946986,18.0,-1.902867,1.902867


Finally we save those lists using only the top 20 scoring pairs to *.csv files for exchange / further investigation.

In [11]:
fout = "./output/ylip_targets_top_20_delta_ctrl_osc.csv"
df_ctrl_v_osc.head(20).to_csv(fout, index=False)